# Visual Anomaly Inspection (Topic 2) — Google Colab Runner

Notebook này đóng vai trò là **bộ điều khiển tự động** trên GPU của Google Colab để chạy toàn bộ pipeline từ thư mục mã nguồn (`scripts/`, `src/`, `configs/`) và lưu kết quả về Google Drive.

In [ ]:
# 1. Kiểm tra GPU & Kết nối Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/VISUAL_ANOMALY_INSPECTION_OUTPUTS'
!mkdir -p {DRIVE_OUT}

In [ ]:
# 2. Clone nhánh hoang_ver từ GitHub & Cài đặt thư viện
%cd /content
!rm -rf VISUAL-ANOMALY-INSPECTION
!git clone -b hoang_ver https://github.com/KNV-DSEB/VISUAL-ANOMALY-INSPECTION.git
%cd /content/VISUAL-ANOMALY-INSPECTION
!pip install -q -r requirements.txt

In [ ]:
# 3. Tải dữ liệu MVTec AD (5 categories) & Chạy EDA (D-02)
!python scripts/download_mvtec.py
!python scripts/eda_mvtec.py
!python -m pytest -v

In [ ]:
# 4. Huấn luyện & Dự đoán cho cả 3 phương pháp Baseline: Autoencoder, PatchCore, One-Class (AE, PC, OC)
!python scripts/train_and_predict.py --method all --artifacts-dir {DRIVE_OUT}/artifacts

In [ ]:
# 5. Chạy thêm cấu hình tinh chỉnh PatchCore Refined (I-02)
!python scripts/train_and_predict.py \
    --method patchcore \
    --config configs/experiments/patchcore_refined.yaml \
    --run-name patchcore_refined \
    --artifacts-dir {DRIVE_OUT}/artifacts

In [ ]:
# 6. Tổng hợp bảng điểm so sánh chung, Pixel Localization (AUROC + PRO) & Error Analysis (E-02, E-03, L-01)
!python scripts/evaluate.py \
    --artifacts-dir {DRIVE_OUT}/artifacts \
    --results-dir {DRIVE_OUT}/results

In [ ]:
# 7. Sinh bộ ảnh Localization Gallery so sánh (Input | GT Mask | AE | PatchCore) (L-02)
!python scripts/generate_galleries.py \
    --artifacts-dir {DRIVE_OUT}/artifacts \
    --results-dir {DRIVE_OUT}/results

In [ ]:
# 8. Chạy thí nghiệm độ nhạy số lượng mẫu Normal & Phân tích chi phí False-Alarm (I-01)
!python scripts/run_sensitivity.py \
    --artifacts-dir {DRIVE_OUT}/artifacts \
    --results-dir {DRIVE_OUT}/results

In [ ]:
# 9. Đóng gói kết quả nhẹ (bảng CSV + hình vẽ) để tải về máy local
!cp -r results/eda_* {DRIVE_OUT}/results/ 2>/dev/null || true
!cd {DRIVE_OUT} && zip -r results_package.zip results/
print('Hoàn tất! Tải file results_package.zip và thư mục artifacts/checkpoints trong Google Drive của bạn.')